# Adaptive Laya Worker

- `WORKER_ID` secret: comma-separated list (e.g. `0,1` runs workers 0 and 1 in parallel on both T4 GPUs).
- **Quick pipeline test**: set `ADAPTIVE_LAYA_TEST` secret to `1` — ingests only `coding,math` capped at 600 records/worker, 1 epoch, **no publish**. Full run in a few minutes.
- Internet ON, GPU T4 x2.


In [ ]:
import os
from pathlib import Path


def _secret(name: str, default: str = "") -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name) or default
    except Exception:  # noqa: BLE001
        return os.environ.get(name, default)

WORKER_IDS = [int(x) for x in _secret("WORKER_ID", "0,1").replace(" ", "").split(",") if x != ""]
MODE = _secret("ADAPTIVE_LAYA_MODE", "probe")
ROUND = int(_secret("ADAPTIVE_LAYA_ROUND", "0"))
REPO_URL = _secret("ADAPTIVE_LAYA_REPO")
GITHUB_TOKEN = _secret("GITHUB_TOKEN")
COMMIT = _secret("ADAPTIVE_LAYA_COMMIT")
HF_TOKEN = _secret("HF_TOKEN")
EPOCHS = _secret("ADAPTIVE_LAYA_EPOCHS", "10")
TEST = _secret("ADAPTIVE_LAYA_TEST", "0").lower() in ("1", "true", "yes")
if TEST:
    EPOCHS = "1"
_repo = "/kaggle/working/repo"
_suffix = "_test" if TEST else ""
_shard_dir = f"data/processed/shards{_suffix}"
_raw_root = "data/raw/hf_test" if TEST else "data/raw"
if os.environ.get("ADAPTIVE_LAYA_LOCAL_DIR"):
    print("WARNING: ADAPTIVE_LAYA_LOCAL_DIR is set in this environment - publishes will go "
          "to a local folder instead of Kaggle! Unset it before running.")
print(WORKER_IDS, MODE, ROUND, "epochs=", EPOCHS, "TEST=", TEST)
if Path(_repo).exists():
    os.chdir(_repo)


In [ ]:
import subprocess
import sys
from pathlib import Path

_repo_abs = "/kaggle/working/repo"
if REPO_URL and not Path(_repo_abs + "/configs/worker.yaml").exists():
    clone_url = REPO_URL
    if GITHUB_TOKEN and REPO_URL.startswith("https://github.com/"):
        clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    res = subprocess.run(["git", "clone", clone_url, _repo_abs], capture_output=True, text=True, check=False)
    if res.returncode != 0:
        print("CLONE STDOUT:", res.stdout)
        print("CLONE STDERR:", res.stderr)
    os.chdir(_repo_abs)
    if COMMIT:
        subprocess.run(["git", "checkout", COMMIT], check=True)
    req = Path(_repo_abs) / "requirements.txt"
    if req.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)


# repo exists -> pull latest (so fixes are deployed)
if Path(_repo_abs + "/configs/worker.yaml").exists():
    os.chdir(_repo_abs)
    if GITHUB_TOKEN and REPO_URL.startswith("https://github.com/"):
        _pull_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
        subprocess.run(["git", "remote", "set-url", "origin", _pull_url], check=False)
    subprocess.run(["git", "fetch", "origin"], check=False)
    if COMMIT:
        subprocess.run(["git", "checkout", COMMIT], check=True)
    else:
        subprocess.run(["git", "pull"], check=False)

# Kaggle's image ships torchao 0.10.0; peft's LoRA injection hard-fails on
# torchao <0.16 even though we never use it - remove it so the peft check is False.
_un = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
                     capture_output=True, text=True, check=False)
if "Successfully uninstalled" in (_un.stdout + _un.stderr):
    print("removed preinstalled torchao (incompatible with peft LoRA; unused by this repo)")

# prefetch the backbone so probe never stalls mid-load on the network
try:
    from huggingface_hub import snapshot_download
    print("backbone cached:", snapshot_download("Qwen/Qwen2.5-Coder-0.5B-Instruct"))
except Exception as _e:  # noqa: BLE001
    print("backbone prefetch failed:", _e)

# restore shards from the saved Kaggle dataset before falling back to a full ingest
_shards = [Path(f"{_shard_dir}/shard-{w}.jsonl") for w in WORKER_IDS]
if not TEST and not all(p.exists() for p in _shards):
    try:
        import shutil as _shutil

        import kagglehub as _kh
        import yaml as _yaml
        _shard_slug = _yaml.safe_load(Path("configs/worker.yaml").read_text(encoding="utf-8"))["datasets"]["shards"]
        _src = Path(_kh.dataset_download(_shard_slug))
        if (_src / "shards.manifest.json").exists():
            Path(_shard_dir).mkdir(parents=True, exist_ok=True)
            for _f in sorted(_src.glob("*")):
                if _f.is_file():
                    _shutil.copy2(_f, Path(_shard_dir) / _f.name)
            print(f"restored shards from {_shard_slug}")
    except Exception as _e:  # noqa: BLE001
        print("shard restore failed (falling back to ingest):", _e)

# ingest + shard if any requested shard is missing
if not all(p.exists() for p in _shards):
    _ingest_cmd = [sys.executable, "scripts/data_generation/ingest_real.py", "--config", "configs/worker.yaml"]
    _ingest_env = {**os.environ, "HF_TOKEN": HF_TOKEN}
    if TEST:
        _ingest_cmd += ["--max-records", "600", "--domains", "coding,math"]
        _ingest_env["ADAPTIVE_LAYA_HF_CACHE"] = "data/raw/hf_test"
        print("TEST mode: ingesting coding+math only, max 600 records each")
    subprocess.run(_ingest_cmd, env=_ingest_env, check=True)
    import json as _json
    all_records = []
    for f in sorted(Path(_raw_root).rglob("decisions.jsonl")):
        for line in f.read_text(encoding="utf-8").splitlines():
            if line.strip():
                all_records.append(_json.loads(line))
    print(f"merged {len(all_records)} records")
    merged = Path(f"data/processed/merged{_suffix}.jsonl")
    merged.parent.mkdir(parents=True, exist_ok=True)
    merged.write_text("\n".join(_json.dumps(r, sort_keys=True, allow_nan=False) for r in all_records) + "\n", encoding="utf-8")
    subprocess.run([sys.executable, "scripts/make_shards.py", "--input", str(merged), "--out", _shard_dir, "--n", "3"], check=True)


In [ ]:
# Feature extraction (probe mode) - one subprocess per GPU
import os
import subprocess
import sys
from pathlib import Path


def _existing_procs(needle):
    """PIDs of live processes whose /proc cmdline contains needle (orphan detection)."""
    pids = []
    try:
        for d in Path("/proc").iterdir():
            if not d.name.isdigit() or int(d.name) == os.getpid():
                continue
            try:
                cmd = (d / "cmdline").read_bytes().decode("utf-8", "replace")
            except OSError:
                continue
            if needle in cmd:
                pids.append(d.name)
    except OSError:
        pass
    return pids


if MODE == "probe":
    _stale = _existing_procs("scripts/training/probe_decoder.py")
    if _stale:
        raise RuntimeError(
            f"probe_decoder.py already running (pids {_stale}) - an earlier run was never stopped. "
            "Duplicate runs oversubscribe both GPUs and freeze the notebook. "
            "Stop the session (or run `!pkill -9 -f probe_decoder.py`) and re-run this cell once."
        )
    procs = []
    for idx, wid in enumerate(WORKER_IDS):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(idx % 2), "HF_TOKEN": HF_TOKEN,
               "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}
        p = subprocess.Popen(
            [sys.executable, f"{_repo}/scripts/training/probe_decoder.py",
             "--input", f"{_repo}/{_shard_dir}/shard-{wid}.jsonl",
             "--out", f"{_repo}/results/features-{wid}{_suffix}.pt"],
            env=env)
        procs.append((wid, p))
    results = [(wid, p.wait()) for wid, p in procs]
    failed = [wid for wid, rc in results if rc != 0]
    missing = [wid for wid in WORKER_IDS if not Path(f"{_repo}/results/features-{wid}{_suffix}.pt").exists()]
    if failed or missing:
        raise RuntimeError(
            f"probe failed: exit codes={results}, missing feature files={missing}. "
            "Check the output above for the probe error (model download / CUDA / shard)."
        )
    print("probe ok:", [(wid, str(Path(f"{_repo}/results/features-{wid}{_suffix}.pt").stat().st_size // 1024) + " KB") for wid in WORKER_IDS])


In [ ]:
# Train (+ publish in full mode) - one subprocess per GPU
# Output goes to results/train-<wid>_<MODE><suffix>.log; this cell tails it every 2 min.
# NEVER re-run this cell while a previous run is still active - use the guard below.
import os
import subprocess
import sys
import time
from pathlib import Path


def _existing_procs(needle):
    """PIDs of live processes whose /proc cmdline contains needle (orphan detection)."""
    pids = []
    try:
        for d in Path("/proc").iterdir():
            if not d.name.isdigit() or int(d.name) == os.getpid():
                continue
            try:
                cmd = (d / "cmdline").read_bytes().decode("utf-8", "replace")
            except OSError:
                continue
            if needle in cmd:
                pids.append(d.name)
    except OSError:
        pass
    return pids


def _log_path(wid):
    return Path(f"{_repo}/results/train-{wid}_{MODE}{_suffix}.log")


def _log_tail(wid, width=110):
    try:
        txt = _log_path(wid).read_text(encoding="utf-8", errors="replace")
    except OSError:
        return "(no log yet)"
    lines = [ln for ln in txt.replace("\r", "\n").split("\n") if ln.strip()]
    if not lines:
        return "(no output yet)"
    return lines[-1][-width:]


def _log_dump(wid, n=15):
    try:
        txt = _log_path(wid).read_text(encoding="utf-8", errors="replace")
    except OSError:
        print(f"----- no log for worker {wid} -----")
        return
    lines = [ln for ln in txt.replace("\r", "\n").split("\n") if ln.strip()]
    print(f"----- tail {_log_path(wid).name} ({len(lines)} lines) -----")
    print("\n".join(lines[-n:]))
    print("----- end tail -----")


_stale = _existing_procs("scripts/training/train_worker.py")
if _stale:
    raise RuntimeError(
        f"train_worker.py already running (pids {_stale}) - a previous run was never stopped. "
        "Re-running this cell while a train is active is what froze the notebook before "
        "(duplicate workers on the same GPUs + orphaned processes blocked writing to a "
        "dead stdout pipe). Stop the session (or run `!pkill -9 -f train_worker.py`) "
        "and re-run this cell exactly once."
    )

if MODE == "probe":
    _missing = [wid for wid in WORKER_IDS if not Path(f"{_repo}/results/features-{wid}{_suffix}.pt").exists()]
    if _missing:
        raise RuntimeError(f"features for workers {_missing} not found - the probe cell must have failed; see its output above")
else:
    _missing = [wid for wid in WORKER_IDS if not Path(f"{_repo}/{_shard_dir}/shard-{wid}.jsonl").exists()]
    if _missing:
        raise RuntimeError(f"shards for workers {_missing} not found - the ingest cell must have failed; see its output above")

Path(f"{_repo}/results").mkdir(parents=True, exist_ok=True)
logs = {}
procs = []
for idx, wid in enumerate(WORKER_IDS):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(idx % 2),
           "HF_TOKEN": HF_TOKEN,
           "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}
    cmd = [sys.executable, f"{_repo}/scripts/training/train_worker.py",
           "--worker-id", str(wid), "--round", str(ROUND), "--mode", MODE,
           "--out", f"{_repo}/results/worker_out_{wid}_{MODE}{_suffix}",
           "--skip-ingest"]
    if not TEST:
        cmd.append("--publish")
    if MODE == "probe":
        cmd += ["--features", f"{_repo}/results/features-{wid}{_suffix}.pt"]
        cmd += ["--epochs", EPOCHS]
    else:
        cmd += ["--shard", f"{_repo}/{_shard_dir}/shard-{wid}.jsonl"]
    logs[wid] = open(_log_path(wid), "w", encoding="utf-8")  # noqa: SIM115 - handle must outlive the Popen spawn loop
    procs.append((wid, subprocess.Popen(cmd, env=env, stdout=logs[wid], stderr=subprocess.STDOUT)))

print(f"started workers {[wid for wid, _ in procs]} - output: results/train-<wid>_{MODE}{_suffix}.log (tail every 2 min)")
_start = time.time()
_last = time.time()
_done = {}
while len(_done) < len(procs):
    if time.time() - _last >= 120:
        _last = time.time()
        _pending = [wid for wid, _ in procs if wid not in _done]
        _tails = " | ".join(f"w{wid}: {_log_tail(wid)}" for wid in _pending)
        print(f"[{int(time.time() - _start)}s] {_tails}", flush=True)
    time.sleep(10)
    for wid, p in procs:
        if wid in _done:
            continue
        rc = p.poll()
        if rc is not None:
            _done[wid] = rc
            print(f"w{wid}: exited rc={rc} after {int(time.time() - _start)}s", flush=True)
for wid, fh in logs.items():
    fh.close()

failed = [wid for wid, rc in _done.items() if rc != 0]
if failed:
    for wid in failed:
        _log_dump(wid, 15)
    raise RuntimeError(f"train failed for workers {failed} - log tails above (full logs: results/train-<wid>_{MODE}{_suffix}.log)")
print("train ok:", dict(_done))


In [ ]:
# Evaluate routers against phase0 gates (in-sample; held-out eval is a separate step)
import json as _json
import shutil as _shutil
import subprocess as _sp
import sys as _sys

_sys.path.insert(0, _repo)
import kagglehub as _kh
import yaml as _yaml

_cfg = _yaml.safe_load(Path("configs/worker.yaml").read_text(encoding="utf-8"))
if MODE == "probe":
    _missing = [w for w in WORKER_IDS if not Path(f"{_repo}/results/features-{w}{_suffix}.pt").exists()]
    if _missing:
        try:
            _src = Path(_kh.dataset_download(_cfg["datasets"]["features"]))
            for w in WORKER_IDS:
                _f = _src / f"features-{w}{_suffix}.pt"
                _d = Path(f"{_repo}/results/features-{w}{_suffix}.pt")
                if _f.exists() and not _d.exists():
                    _shutil.copy2(_f, _d)
            print("features restored from", _cfg["datasets"]["features"])
        except Exception as _e:  # noqa: BLE001
            print("feature restore failed:", _e)
    for wid in WORKER_IDS:
        _feat = f"{_repo}/results/features-{wid}{_suffix}.pt"
        _out = f"{_repo}/results/worker_out_{wid}_{MODE}{_suffix}"
        if not Path(_feat).exists() or not Path(f"{_out}/router.pt").exists():
            print(f"worker {wid}: features/router missing - skip eval")
            continue
        r = _sp.run([_sys.executable, "-m", "scripts.evaluation.evaluate_router",
                     "--features", _feat,
                     "--router", f"{_out}/router.pt",
                     "--router-config", f"{_out}/router_config.json",
                     "--out", f"{_repo}/results/router_eval_{wid}{_suffix}.json",
                     "--gates", f"{_repo}/configs/phase0_qwen.yaml"],
                    cwd=_repo, capture_output=True, text=True, check=False)
        if r.returncode != 0:
            print(f"worker {wid} eval FAILED\n{r.stdout}\n{r.stderr}")
        else:
            res = _json.loads(r.stdout)
            print(f"worker {wid}: passed={res['passed']} checks={res['checks']}")
            print("  ", res["metrics"])
else:
    print("eval cell runs in probe mode only")


In [ ]:
import sys as _sys

_sys.path.insert(0, _repo)
from scripts.kaggle_io import local_mode

if TEST:
    print("TEST run complete - checkpoints NOT published (set ADAPTIVE_LAYA_TEST=0 for a full run)")
elif local_mode():
    print("workers done - WARNING: publish wrote to local folder repo/1 (session-only), NOT to Kaggle")
else:
    print("workers done - checkpoints published to Kaggle datasets")
